# 28. Text-Image In-Batch — Remaining Sizes (Train-or-Resume to Target Epochs)



Each size is driven to `TARGET_EPOCHS`. On entry to `resume_size_to_target_epochs`, the cases are:

| State on disk | Behavior |
|---|---|
| `size_{N}/gt_results.csv` **exists** | **Skip** — already complete. Return the loaded GT summary so it still appears in the combined rollup. |
| `size_{N}/latest_checkpoint.pt` **exists**, epoch < target | **Resume** from the checkpoint. |
| `size_{N}/latest_checkpoint.pt` **exists**, epoch ≥ target | **Finalize** — no more training, just (re)write `gt_results.csv`, `training_history.csv`, and per-model `.pt` files. |
| `size_{N}/latest_checkpoint.pt` **missing** | **Train from scratch** (fresh weights, fresh optimizer, epoch 0 → target). **No error, no skip.** |


### Speedup applied (this version)

**DINOv3 image encoding is batched + chunked** (`IMAGE_ENCODE_CHUNK_SIZE = 16`), matching notebook 29.

**Text-image BGE limit: 512 tokens.** All NB28 image/text sizes, including the full-pool run, use this limit. GT evaluation batches examples in groups of `GT_EVAL_CHUNK_SIZE = 16` to bound memory; this batching does not change the token limit.

### Output root

**All outputs** — log file, per-size results, per-size checkpoints, the full-size E3 checkpoint parent, the combined `summary_all_sizes.csv` — live under:

```
OUTPUT_ROOT = results/runs/section_4
```

Inputs (jsonl pairs, leak flags, FN pickles, serializer, GT files) are **unchanged** — they remain in their original source locations.


In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## Section 0 — Installations

Same pin set as notebook 29 (needed for DINOv3 + `cache_dir`-based loading).

In [1]:
# !pip install openpyxl
# !pip install pandas --upgrade
# !pip install accelerate
# !pip install transformers==4.57.6   # downgrade for DINOv3
# !pip install --upgrade huggingface_hub

### dinov3 is a gated model — ask for grant on HuggingFace, then log in once before the first download.

In [2]:
# from huggingface_hub import login
# login()

## Section 1 — Setup, logging, config

Logging: notebook 29's single shared file + logger name `img_inbatch_ft`, rooted under `OUTPUT_ROOT`.
HF cache: notebook 29's explicit `HF_CACHE` variable, passed as `cache_dir=` everywhere.
Pickle helpers: notebook 29's `pkl_vars` / `reload_vars`, plus a `load_or_build` convenience wrapper.

In [1]:
import json
import os
import re
import sys
import time
import math
import random
import pickle
import logging
import gc
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer

import transformers
import openpyxl  # noqa: F401  (kept for parity with notebook 29)

In [2]:
transformers.logging.set_verbosity_error()

# --- Output root: EVERY output (logs, results, checkpoints) lives under here ---
OUTPUT_ROOT = str(RESULTS_RUNS_ROOT / 'size_sweeps')
os.makedirs(OUTPUT_ROOT, exist_ok=True)

LOG_FORMAT = '%(asctime)s : %(levelname)s : %(message)s'
LOG_FILE   = os.path.join(OUTPUT_ROOT, "img_inbatch_ft.log")

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("img_inbatch_ft")

# Notebook 29 logs file-only. Add a stdout echo so progress is visible in the notebook
# output too (idempotent — only adds the StreamHandler once).
if not any(type(h) is logging.StreamHandler for h in logger.handlers):
    _sh = logging.StreamHandler(sys.stdout)
    _sh.setFormatter(logging.Formatter(LOG_FORMAT))
    logger.addHandler(_sh)


if torch.cuda.device_count() > 1:
    torch.cuda.set_device(1)
    CUDA="cuda:0" # cuda for NB 28, cuda:1 for NB 28B
else: 
    CUDA = "cuda:0"
device = CUDA if torch.cuda.is_available() else "cpu"
logger.info("Device: %s", device)
if device == CUDA:
    logger.info("GPU: %s", torch.cuda.get_device_name(torch.cuda.current_device()))


# --- Pickle helpers (notebook 29's pkl_vars / reload_vars, plus a load_or_build wrapper) ---
VARS_DIR = str(FINETUNING_ROOT / "vars")            # same folder notebook 29 writes to
os.makedirs(VARS_DIR, exist_ok=True)

def pkl_vars(varname, filename):
    with open(filename, 'wb') as file:
        pickle.dump(varname, file)

def reload_vars(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

def load_or_build(filename, builder_fn, label=""):
    """Read pickle if it exists, else run builder_fn() and cache the result."""
    if os.path.exists(filename):
        logger.info("[%s] Loaded cached vars: %s", label, filename)
        return reload_vars(filename)
    logger.info("[%s] No cache at %s -- building from source", label, filename)
    val = builder_fn()
    pkl_vars(val, filename)
    logger.info("[%s] Cached to %s", label, filename)
    return val

2026-10-24 11:26:54,107 : INFO : Device: cuda:0
2026-10-24 11:26:54,110 : INFO : GPU: Quadro RTX 8000


### Config

Two independent sets of constants (image/text vs. text/table), plus notebook 29's explicit `HF_CACHE`.
All **output** paths (logs, results, checkpoints) are built from `OUTPUT_ROOT`; all **input** paths are unchanged.

Two chunk constants, kept separate on purpose:
- **`IMAGE_ENCODE_CHUNK_SIZE = 16`** — DINOv3 image forward passes (train + GT eval).
- **`GT_EVAL_CHUNK_SIZE = 16`** — BGE text forward passes **during GT eval only** are batched in groups of 16 examples; text inputs remain limited to `IMG_BGE_MAX_LENGTH = 512`.

One more constant now: **`_MODEL_FILE_RENAMES`** maps the in-memory state-dict keys to NB29's on-disk filenames, so the artefacts are byte-for-byte named the same as notebook 29.

In [3]:
# --- Shared ---
HF_CACHE          = DEFAULT_HF_CACHE
RANDOM_SEED       = 42
TARGET_EPOCHS     = 20
CHECKPOINT_EVERY_N_STEPS = 500
MAX_RUNTIME_HOURS_PER_SIZE = 10.0
K_VALUES          = [1, 3, 5, 10]
HEAD_LEARNING_RATE = 1e-4
BGE_ENCODER_LR    = 1e-5
TEMPERATURE       = 0.07
N_UNFROZEN_LAYERS_BGE = 6
SHARED_DIM        = 512
BGE_MODEL_NAME    = "BAAI/bge-m3"
FN_SIMILARITY_THRESHOLD = 0.8

# --- Chunking constants (see markdown note above) ---
IMAGE_ENCODE_CHUNK_SIZE = 16   # batch+chunk DINOv3 image encoding (train + GT eval)
GT_EVAL_CHUNK_SIZE      = 16   # batch BGE text encoding during GT eval

# --- Model-file renaming: state-dict key -> on-disk filename stem (matches NB29 exactly) ---
_MODEL_FILE_RENAMES = {
    "dinov3_model": "dinov3_model_unfrozen",
    "bge_model":    "bge_model_unfrozen",
    # head_* keys are unchanged:
    # head_text_bge -> head_text_bge.pt
    # head_image_dinov3 -> head_image_dinov3.pt
    # head_text_bge_v2 -> head_text_bge_v2.pt
    # head_table_bge_v2 -> head_table_bge_v2.pt
}

# --- INPUT paths (unchanged — source data lives where it always did) ---
FN_METADATA_PATH  = str(FINETUNING_ROOT / 'Shared_data' / 'df_combined_metadata.pkl')
IMG_BASE_DIR      = str(BSV_ROOT / 'Automatic Construction')
TEXTIMAGE_PAIRS_PATH = os.path.join(IMG_BASE_DIR, "bsv_pairs_auto_textimage.jsonl")
IMAGE_DIR         = os.path.join(IMG_BASE_DIR, "bsv_images_auto_textimage") # the images are not published, but can be rehydrated
LEAK_FLAGS_PATH   = str(FINETUNING_ROOT / 'Shared_data' / 'table_leak_flags_COMPLETE.xlsx')
FN_IMAGE_IMAGE_PATH = str(FINETUNING_ROOT / 'Shared_data' / 'reranked_image_image.pkl')
DINOV3_MODEL_NAME = "facebook/dinov3-vitb16-pretrain-lvd1689m"
N_UNFROZEN_LAYERS_DINOV3 = 6
DINOV3_ENCODER_LR = 1e-6
IMG_BATCH_SIZE    = 32
IMG_BGE_MAX_LENGTH = 512
GT_TEXTIMAGE_PATH = str(BSV_ROOT / 'Ground Truth' / 'bsv_gt_textimage_final_500.jsonl')
GT_IMAGE_DIR      = str(BSV_ROOT / 'Ground Truth' / 'bsv_images_gt_2404')

TAB_BASE_DIR      = str(BSV_ROOT)
TEXTTABLE_PAIRS_PATH = os.path.join(TAB_BASE_DIR, "Automatic Construction", "bsv_pairs_auto_texttable.jsonl")
SERIALIZER_PATH   = str(FINETUNING_ROOT / 'Shared_data')
FN_TABLE_TABLE_PATH = str(FINETUNING_ROOT / 'Shared_data' / 'reranked_table_table.pkl')
TAB_BATCH_SIZE    = 32# 16
TAB_BGE_MAX_LENGTH = 1024 # was 2048, but 1024 cover already more than 90% tables
GT_TEXTTABLE_PATH = str(BSV_ROOT / 'Ground Truth' / 'bsv_gt_texttable_final_merged.jsonl')

# --- OUTPUT paths (all under OUTPUT_ROOT) ---
IMG_OUTPUT_DIR    = os.path.join(OUTPUT_ROOT, "TextImage_Smaller_train_sizes_runs")
TAB_OUTPUT_DIR    = os.path.join(OUTPUT_ROOT, "Texttable_Smaller_train_sizes_runs")
IMG_SIZES_TO_RESUME = [100, 500, 1000, 2500, 5000, 10000, 20000, 33526]  
TAB_SIZES_TO_RESUME = [100, 500, 1000, 2500, 5000, 6757]           

# --- Full-size text/table checkpoint parent (under OUTPUT_ROOT) ---
E3_FULL_CKPT_PARENT_DIR = os.path.join(OUTPUT_ROOT, "")
E3_FULL_CKPT_SUBDIR_NAME = "checkpoints"

# Ensure output roots exist (resume also makedirs size_N, but make the parent layout up-front).
for _p in (IMG_OUTPUT_DIR, TAB_OUTPUT_DIR):
    os.makedirs(_p, exist_ok=True)

logger.info("Config loaded. HF_CACHE=%s, OUTPUT_ROOT=%s, VARS_DIR=%s", HF_CACHE, OUTPUT_ROOT, VARS_DIR)
logger.info("Chunk sizes: IMAGE_ENCODE_CHUNK_SIZE=%d, GT_EVAL_CHUNK_SIZE=%d",
            IMAGE_ENCODE_CHUNK_SIZE, GT_EVAL_CHUNK_SIZE)
logger.info("TEXTIMAGE_PAIRS_PATH exists: %s", os.path.exists(TEXTIMAGE_PAIRS_PATH))
logger.info("TEXTTABLE_PAIRS_PATH exists: %s", os.path.exists(TEXTTABLE_PAIRS_PATH))
logger.info("IMG_OUTPUT_DIR exists: %s", os.path.exists(IMG_OUTPUT_DIR))
logger.info("TAB_OUTPUT_DIR exists: %s", os.path.exists(TAB_OUTPUT_DIR))
logger.info("E3_FULL_CKPT_PARENT_DIR exists: %s", os.path.exists(E3_FULL_CKPT_PARENT_DIR))

2026-10-24 11:26:54,141 : INFO : Config loaded. HF_CACHE=/projects/agrifutur/hf_cache2, OUTPUT_ROOT=/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes, VARS_DIR=vars
2026-10-24 11:26:54,143 : INFO : Chunk sizes: IMAGE_ENCODE_CHUNK_SIZE=16, GT_EVAL_CHUNK_SIZE=16
2026-10-24 11:26:54,146 : INFO : TEXTIMAGE_PAIRS_PATH exists: True
2026-10-24 11:26:54,148 : INFO : TEXTTABLE_PAIRS_PATH exists: True
2026-10-24 11:26:54,150 : INFO : IMG_OUTPUT_DIR exists: True
2026-10-24 11:26:54,152 : INFO : TAB_OUTPUT_DIR exists: True
2026-10-24 11:26:54,154 : INFO : E3_FULL_CKPT_PARENT_DIR exists: True


## Section 2 — Load data for both pair types

Reproduces each original notebook's pool-building **exactly** (image/text leak-filters, text/table does not), but wraps every artefact in `load_or_build` so the jsonl is only parsed once — subsequent runs are pure pickle reloads, matching notebook 29's pattern.

In [4]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

sys.path.insert(0, SERIALIZER_PATH)
from serializer import serialize_table_html_v3

# --- Image/text: raw pairs (shared with notebook 29 -- same pickle file) ---
raw_textimage = load_or_build(
    os.path.join(VARS_DIR, "raw_textimage.pkl"),
    lambda: load_jsonl(TEXTIMAGE_PAIRS_PATH),
    label="raw_textimage",
)


# --- Image/text: pool (NOT notebook 29's textimage_records.pkl -- schema and pool size differ) ---
# Notebook 29's pickle drops section_title (which the FN masks here need) and applies a
# bulletin_meta filter (33208 records vs. this notebook's 33526 = leak-filter only).
def _build_textimage_records():
    records = []
    for i, r in enumerate(raw_textimage):
        if r.get("pair_type") != "text_image":
            continue
        text = r.get("text", "").strip()
        filename = r.get("image", {}).get("filename")
        if not text or not filename:
            continue
        records.append({
            "orig_idx": i, "bulletin_id": r.get("bulletin_id"),
            "section_title": r.get("section_title"),
            "text": text, "image_path": os.path.join(IMAGE_DIR, filename),
        })
    before_count = len(records)
    leak_df = pd.read_excel(LEAK_FLAGS_PATH)
    leaked_indices = set(leak_df["row_index"])
    records = [r for r in records if r["orig_idx"] not in leaked_indices]
    logger.info("Image/text: %d -> %d after leak-filter", before_count, len(records))
    return records

textimage_records = load_or_build(
    os.path.join(VARS_DIR, "nb28_textimage_records.pkl"),
    _build_textimage_records,
    label="nb28_textimage_records",
)
assert len(textimage_records) == 33526, f"Image/text pool mismatch: got {len(textimage_records)}, expected 33526"


# --- Text/table: raw pairs (notebook 29 never touched this side -- new pickle) ---
raw_texttable = load_or_build(
    os.path.join(VARS_DIR, "raw_texttable.pkl"),
    lambda: load_jsonl(TEXTTABLE_PAIRS_PATH),
    label="raw_texttable",
)


def _build_texttable_records():
    records = []
    n_serialize_errors = 0
    for i, r in enumerate(raw_texttable):
        text = r.get("text", "").strip()
        html = r.get("table", {}).get("table_html", "")
        if not text or not html:
            continue
        try:
            serialized = serialize_table_html_v3(html)
        except Exception:
            n_serialize_errors += 1
            continue
        if not serialized.strip():
            continue
        records.append({
            "orig_idx": i, "bulletin_id": r.get("bulletin_id"),
            "section_title": r.get("section_title"),
            "text": text, "table_text": serialized,
        })
    logger.info("Text/table: %d usable records (%d serialize errors, no leak-filter)",
                len(records), n_serialize_errors)
    return records

texttable_records = load_or_build(
    os.path.join(VARS_DIR, "texttable_records.pkl"),
    _build_texttable_records,
    label="texttable_records",
)
assert len(texttable_records) == 6757, f"Text/table pool mismatch: got {len(texttable_records)}, expected 6757"

2026-10-24 11:26:54,232 : INFO : [raw_textimage] Loaded cached vars: vars/raw_textimage.pkl
2026-10-24 11:26:54,509 : INFO : [nb28_textimage_records] Loaded cached vars: vars/nb28_textimage_records.pkl
2026-10-24 11:26:54,604 : INFO : [raw_texttable] Loaded cached vars: vars/raw_texttable.pkl
2026-10-24 11:26:54,662 : INFO : [texttable_records] Loaded cached vars: vars/texttable_records.pkl


## Section 3 — False-negative masks (image-image and table-table, unchanged from originals)

In [5]:
def build_key(bulletin_id, section_title, text):
    return (bulletin_id, section_title, text)

def build_fn_mask_set(pkl_path, threshold=FN_SIMILARITY_THRESHOLD):
    df = pd.read_pickle(pkl_path)
    hi = df[df["colbert"] >= threshold]
    pairs = set()
    for _, row in hi.iterrows():
        pairs.add(frozenset({int(row["global_i"]), int(row["global_j"])}))
    return pairs

fn_metadata = pd.read_pickle(FN_METADATA_PATH)

# --- Image-image mask ---
fn_image_image = build_fn_mask_set(FN_IMAGE_IMAGE_PATH)
key_to_globals_image = defaultdict(list)
for _, row in fn_metadata[fn_metadata["origin"] == "text_image"].iterrows():
    k = build_key(row["bulletin_id"], row["section_title"], row["text"])
    key_to_globals_image[k].append(row["_global"])
logger.info("Image-image: %d flagged pairs, %d unique keys", len(fn_image_image), len(key_to_globals_image))

def is_masked_image_image(record_i, record_j, threshold_set=fn_image_image):
    key_i = build_key(record_i["bulletin_id"], record_i["section_title"], record_i["text"])
    key_j = build_key(record_j["bulletin_id"], record_j["section_title"], record_j["text"])
    for gi in key_to_globals_image.get(key_i, []):
        for gj in key_to_globals_image.get(key_j, []):
            if frozenset({gi, gj}) in threshold_set:
                return True
    return False

# --- Table-table mask ---
fn_table_table = build_fn_mask_set(FN_TABLE_TABLE_PATH)
key_to_globals_table = defaultdict(list)
for _, row in fn_metadata[fn_metadata["origin"] == "text_table"].iterrows():
    k = build_key(row["bulletin_id"], row["section_title"], row["text"])
    key_to_globals_table[k].append(row["_global"])
logger.info("Table-table: %d flagged pairs, %d unique keys", len(fn_table_table), len(key_to_globals_table))

def is_masked_table_table(record_i, record_j, threshold_set=fn_table_table):
    key_i = build_key(record_i["bulletin_id"], record_i["section_title"], record_i["text"])
    key_j = build_key(record_j["bulletin_id"], record_j["section_title"], record_j["text"])
    for gi in key_to_globals_table.get(key_i, []):
        for gj in key_to_globals_table.get(key_j, []):
            if frozenset({gi, gj}) in threshold_set:
                return True
    return False

logger.info("Both FN masking functions ready.")

2026-10-24 11:27:01,372 : INFO : Image-image: 240959 flagged pairs, 27476 unique keys
2026-10-24 11:27:02,725 : INFO : Table-table: 55459 flagged pairs, 6561 unique keys
2026-10-24 11:27:02,728 : INFO : Both FN masking functions ready.


## Section 4 — Model builders

Both use **`cache_dir=HF_CACHE`** (notebook 29 convention) on every `from_pretrained(...)` call.

**Speedup:** `build_models_imagetext` exposes `encode_dinov3_images_batched`, mirroring notebook 29.

In [6]:
def build_models_imagetext():
    dinov3_processor = AutoImageProcessor.from_pretrained(DINOV3_MODEL_NAME, cache_dir=HF_CACHE)
    dinov3_model = AutoModel.from_pretrained(DINOV3_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    for p in dinov3_model.parameters():
        p.requires_grad = False
    for layer in dinov3_model.layer[-N_UNFROZEN_LAYERS_DINOV3:]:
        for p in layer.parameters():
            p.requires_grad = True
    DINOV3_DIM = dinov3_model.config.hidden_size

    def encode_dinov3_images_batched(image_paths, chunk_size=IMAGE_ENCODE_CHUNK_SIZE):
        """Batched + chunked DINOv3 image encoding. Returns (len(image_paths), DINOV3_DIM)."""
        if len(image_paths) <= chunk_size:
            images = [Image.open(p).convert("RGB") for p in image_paths]
            inputs = dinov3_processor(images=images, return_tensors="pt").to(device)
            out = dinov3_model(**inputs)
            return out.last_hidden_state[:, 0, :]
        chunks = []
        for i in range(0, len(image_paths), chunk_size):
            batch_paths = image_paths[i:i + chunk_size]
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            inputs = dinov3_processor(images=images, return_tensors="pt").to(device)
            out = dinov3_model(**inputs)
            chunks.append(out.last_hidden_state[:, 0, :])
        return torch.cat(chunks, dim=0)

    bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE)
    bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    for p in bge_model.parameters():
        p.requires_grad = False
    for layer in bge_model.encoder.layer[-N_UNFROZEN_LAYERS_BGE:]:
        for p in layer.parameters():
            p.requires_grad = True
    BGE_DIM = bge_model.config.hidden_size

    def encode_bge_text(texts):
        inputs = bge_tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                                 max_length=IMG_BGE_MAX_LENGTH).to(device)
        out = bge_model(**inputs)
        return out.last_hidden_state[:, 0, :]

    class ProjectionHead(nn.Module):
        def __init__(self, in_dim, out_dim=SHARED_DIM):
            super().__init__()
            self.proj = nn.Linear(in_dim, out_dim)
        def forward(self, x):
            return F.normalize(self.proj(x), p=2, dim=-1)

    head_text_bge = ProjectionHead(BGE_DIM).to(device)
    head_image_dinov3 = ProjectionHead(DINOV3_DIM).to(device)

    optimizer = torch.optim.AdamW([
        {"params": list(head_text_bge.parameters()) + list(head_image_dinov3.parameters()), "lr": HEAD_LEARNING_RATE},
        {"params": [p for p in dinov3_model.parameters() if p.requires_grad], "lr": DINOV3_ENCODER_LR},
        {"params": [p for p in bge_model.parameters() if p.requires_grad], "lr": BGE_ENCODER_LR},
    ])

    return {
        "dinov3_model": dinov3_model, "encode_dinov3_images_batched": encode_dinov3_images_batched,
        "bge_model": bge_model, "encode_bge_text": encode_bge_text,
        "head_text_bge": head_text_bge, "head_image_dinov3": head_image_dinov3,
        "optimizer": optimizer,
    }


def build_models_texttable():
    bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE)
    bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    for p in bge_model.parameters():
        p.requires_grad = False
    for layer in bge_model.encoder.layer[-N_UNFROZEN_LAYERS_BGE:]:
        for p in layer.parameters():
            p.requires_grad = True
    BGE_DIM = bge_model.config.hidden_size

    def encode_bge_text(texts):
        inputs = bge_tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                                 max_length=TAB_BGE_MAX_LENGTH).to(device)
        out = bge_model(**inputs)
        return out.last_hidden_state[:, 0, :]

    class ProjectionHead(nn.Module):
        def __init__(self, in_dim, out_dim=SHARED_DIM):
            super().__init__()
            self.proj = nn.Linear(in_dim, out_dim)
        def forward(self, x):
            return F.normalize(self.proj(x), p=2, dim=-1)

    head_text_bge_v2 = ProjectionHead(BGE_DIM).to(device)
    head_table_bge_v2 = ProjectionHead(BGE_DIM).to(device)

    optimizer = torch.optim.AdamW([
        {"params": list(head_text_bge_v2.parameters()) + list(head_table_bge_v2.parameters()), "lr": HEAD_LEARNING_RATE},
        {"params": [p for p in bge_model.parameters() if p.requires_grad], "lr": BGE_ENCODER_LR},
    ])

    return {
        "bge_model": bge_model, "encode_bge_text": encode_bge_text,
        "head_text_bge_v2": head_text_bge_v2, "head_table_bge_v2": head_table_bge_v2,
        "optimizer": optimizer,
    }

logger.info("Both model builders ready (image encoding now batched+chunked at %d).",
            IMAGE_ENCODE_CHUNK_SIZE)

2026-10-24 11:27:02,745 : INFO : Both model builders ready (image encoding now batched+chunked at 16).


## Section 5 — Loss functions (both unchanged in math from their original notebooks)

In [7]:
def make_batches(records, batch_size, shuffle=True):
    idx = np.arange(len(records))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start:start + batch_size]
        yield [records[i] for i in batch_idx]


def masked_symmetric_infonce(sim_matrix, mask_matrix, temperature=TEMPERATURE):
    B = sim_matrix.shape[0]
    logits = sim_matrix / temperature
    neg_inf = torch.finfo(logits.dtype).min
    mask_matrix = mask_matrix.clone()
    mask_matrix.fill_diagonal_(False)
    logits = logits.masked_fill(mask_matrix, neg_inf)
    targets = torch.arange(B, device=logits.device)
    loss_i2j = F.cross_entropy(logits, targets)
    loss_j2i = F.cross_entropy(logits.T, targets)
    return (loss_i2j + loss_j2i) / 2


def compute_imagetext_loss(batch, models):
    texts = [r["text"] for r in batch]
    image_paths = [r["image_path"] for r in batch]
    B = len(batch)
    text_emb_raw = models["encode_bge_text"](texts)
    image_embs_raw = models["encode_dinov3_images_batched"](image_paths)
    text_emb = models["head_text_bge"](text_emb_raw)
    image_emb = models["head_image_dinov3"](image_embs_raw)
    sim = text_emb @ image_emb.T
    mask = torch.zeros((B, B), dtype=torch.bool, device=device)
    for i in range(B):
        for j in range(B):
            if i != j and is_masked_image_image(batch[i], batch[j]):
                mask[i, j] = True
    return masked_symmetric_infonce(sim, mask)


def compute_texttable_loss(batch, models):
    texts = [r["text"] for r in batch]
    tables = [r["table_text"] for r in batch]
    B = len(batch)
    text_emb_raw = models["encode_bge_text"](texts)
    table_emb_raw = models["encode_bge_text"](tables)
    text_emb = models["head_text_bge_v2"](text_emb_raw)
    table_emb = models["head_table_bge_v2"](table_emb_raw)
    sim = text_emb @ table_emb.T
    mask = torch.zeros((B, B), dtype=torch.bool, device=device)
    for i in range(B):
        for j in range(B):
            if i != j and is_masked_table_table(batch[i], batch[j]):
                mask[i, j] = True
    return masked_symmetric_infonce(sim, mask)

logger.info("Both loss functions ready (image/text uses batched+chunked DINOv3).")

2026-10-24 11:27:02,806 : INFO : Both loss functions ready (image/text uses batched+chunked DINOv3).


## Section 6 — Load GT sets, metrics, evaluation functions (both pair types)

GT artefacts follow the same `load_or_build` pattern:
- Image/text GT reuses notebook 29's `vars/raw_gt_textimage.pkl` and `vars/gt_textimage_records.pkl` (schema **exactly matches**).
- Text/table GT builds its own pickles (notebook 29 never touched this side).

**Eval chunking:**
- **DINOv3 image encoding** — batched + chunked via `encode_dinov3_images_batched`.
- **BGE text encoding** — GT examples are batched via `encode_bge_text_chunked_eval`; each text is truncated to 512 tokens.

In [8]:
# --- Image/text GT (shared pickle files with notebook 29 -- schema is identical) ---
raw_gt_textimage = load_or_build(
    os.path.join(VARS_DIR, "raw_gt_textimage.pkl"),
    lambda: load_jsonl(GT_TEXTIMAGE_PATH),
    label="raw_gt_textimage",
)


def _build_gt_textimage_records():
    records = []
    for i, r in enumerate(raw_gt_textimage):
        text = r.get("text", "").strip()
        filename = r.get("image", {}).get("filename")
        if not text or not filename:
            continue
        img_path = os.path.join(GT_IMAGE_DIR, filename)
        if not os.path.exists(img_path):
            continue
        records.append({"gt_idx": i, "text": text, "image_path": img_path})
    logger.info("GT text/image records: %d", len(records))
    return records

gt_textimage_records = load_or_build(
    os.path.join(VARS_DIR, "gt_textimage_records.pkl"),   # shared with notebook 29
    _build_gt_textimage_records,
    label="gt_textimage_records",
)
assert len(gt_textimage_records) > 0


# --- Text/table GT (new pickles -- notebook 29 never touched this side) ---
raw_gt_texttable = load_or_build(
    os.path.join(VARS_DIR, "raw_gt_texttable.pkl"),
    lambda: load_jsonl(GT_TEXTTABLE_PATH),
    label="raw_gt_texttable",
)


def _build_gt_texttable_records():
    records = []
    for i, r in enumerate(raw_gt_texttable):
        text = r.get("text", "").strip()
        html = r.get("table", {}).get("table_html", "") if isinstance(r.get("table"), dict) else r.get("table_html", "")
        if not text or not html:
            continue
        try:
            serialized = serialize_table_html_v3(html)
        except Exception:
            continue
        if not serialized.strip():
            continue
        records.append({"gt_idx": i, "text": text, "table_text": serialized})
    logger.info("GT text/table records: %d", len(records))
    return records

gt_texttable_records = load_or_build(
    os.path.join(VARS_DIR, "gt_texttable_records.pkl"),
    _build_gt_texttable_records,
    label="gt_texttable_records",
)
assert len(gt_texttable_records) > 0


def compute_retrieval_metrics(sim_matrix, k_values=K_VALUES):
    N = sim_matrix.shape[0]
    true_scores = sim_matrix.diag().unsqueeze(1)
    ranks = (sim_matrix >= true_scores).sum(dim=1)
    results = {}
    for k in k_values:
        results[f"R@{k}"] = (ranks <= k).float().mean().item()
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = N
    return results


# Batch GT text encoding to bound memory; each text uses IMG_BGE_MAX_LENGTH=512.
def encode_bge_text_chunked_eval(texts, models, chunk_size=GT_EVAL_CHUNK_SIZE):
    if len(texts) <= chunk_size:
        return models["encode_bge_text"](texts)
    chunks = []
    for i in range(0, len(texts), chunk_size):
        chunks.append(models["encode_bge_text"](texts[i:i + chunk_size]))
    return torch.cat(chunks, dim=0)


def evaluate_imagetext(models):
    models["head_text_bge"].eval(); models["head_image_dinov3"].eval()
    models["dinov3_model"].eval(); models["bge_model"].eval()
    with torch.no_grad():
        image_embs_raw = models["encode_dinov3_images_batched"](
            [r["image_path"] for r in gt_textimage_records]
        )
        gt_image_emb = models["head_image_dinov3"](image_embs_raw)
        gt_text_emb = models["head_text_bge"](
            encode_bge_text_chunked_eval([r["text"] for r in gt_textimage_records], models)
        )
    sim = gt_image_emb @ gt_text_emb.T
    summary = pd.DataFrame([
        {"direction": "text -> image", **compute_retrieval_metrics(sim.T)},
        {"direction": "image -> text", **compute_retrieval_metrics(sim)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]
    models["head_text_bge"].train(); models["head_image_dinov3"].train()
    models["dinov3_model"].train(); models["bge_model"].train()
    return summary


def evaluate_texttable(models):
    models["head_text_bge_v2"].eval(); models["head_table_bge_v2"].eval(); models["bge_model"].eval()
    with torch.no_grad():
        gt_text_emb = models["head_text_bge_v2"](encode_bge_text_chunked_eval([r["text"] for r in gt_texttable_records], models))
        gt_table_emb = models["head_table_bge_v2"](encode_bge_text_chunked_eval([r["table_text"] for r in gt_texttable_records], models))
    sim = gt_text_emb @ gt_table_emb.T
    summary = pd.DataFrame([
        {"direction": "text -> table", **compute_retrieval_metrics(sim)},
        {"direction": "table -> text", **compute_retrieval_metrics(sim.T)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]
    models["head_text_bge_v2"].train(); models["head_table_bge_v2"].train(); models["bge_model"].train()
    return summary

logger.info("GT loaded, both evaluation functions ready (image batched+chunked, text chunked).")

2026-10-24 11:27:02,842 : INFO : [raw_gt_textimage] Loaded cached vars: vars/raw_gt_textimage.pkl
2026-10-24 11:27:02,850 : INFO : [gt_textimage_records] Loaded cached vars: vars/gt_textimage_records.pkl
2026-10-24 11:27:02,854 : INFO : [raw_gt_texttable] Loaded cached vars: vars/raw_gt_texttable.pkl
2026-10-24 11:27:02,859 : INFO : [gt_texttable_records] Loaded cached vars: vars/gt_texttable_records.pkl
2026-10-24 11:27:02,864 : INFO : GT loaded, both evaluation functions ready (image batched+chunked, text chunked).


## Section 7 — Train-or-resume function (fresh-start if no checkpoint)

Results layout matches notebook 29: `size_dir/` contains `latest_checkpoint.pt`, `training_history.csv`, `gt_results.csv`, and model `.pt` files with NB29's exact names (`dinov3_model_unfrozen.pt`, `bge_model_unfrozen.pt`, plus the head files). **No per-epoch CSV tree** — per-epoch GT is evaluated and **logged** to `img_inbatch_ft.log` every epoch (matching NB29's behaviour), but never materialised as CSVs.

**Train-or-resume decision tree:**
1. `gt_results.csv` exists → skip, return existing summary.
2. `latest_checkpoint.pt` exists, epoch < target → resume.
3. `latest_checkpoint.pt` exists, epoch ≥ target → finalize only (no more training).
4. `latest_checkpoint.pt` missing → **train from scratch**, epoch 0 → target. No error, no skip.

In [9]:
def resume_size_to_target_epochs(N, records_pool, output_dir, batch_size, build_models_fn, loss_fn,
                                   eval_fn, state_dict_keys, loss_key, target_epochs=TARGET_EPOCHS,
                                   label=""):
    """Drive `output_dir/size_{N}` to `target_epochs`. Decision tree (matches notebook 29):

      1. gt_results.csv exists          -> already complete; skip and return loaded summary.
      2. latest_checkpoint.pt exists:
           a. ckpt["epoch"] <  target   -> resume from checkpoint.
           b. ckpt["epoch"] >= target   -> finalize artefacts only (no training loop).
      3. latest_checkpoint.pt missing   -> train FRESH from epoch 0 -> target. No error, no skip.

    Results layout (matches NB29 exactly):
      size_dir/
        latest_checkpoint.pt
        training_history.csv
        gt_results.csv
        <state_dict_keys mapped via _MODEL_FILE_RENAMES>.pt

    Per-epoch GT is evaluated and logged every epoch, but NOT written as CSVs (NB29 parity).
    """
    size_dir = os.path.join(output_dir, f"size_{N}")
    os.makedirs(size_dir, exist_ok=True)          # <-- FIX: create size_dir even on fresh start
    ckpt_path = os.path.join(size_dir, "latest_checkpoint.pt")
    gt_results_path = os.path.join(size_dir, "gt_results.csv")


    # --- Case 1: already complete -> skip ---
    if os.path.exists(gt_results_path):
        logger.info("[%s N=%d] Already completed (%s exists) -- skipping.", label, N, gt_results_path)
        try:
            return pd.read_csv(gt_results_path)
        except Exception as e:
            logger.warning("[%s N=%d] Failed to read existing %s: %s", label, N, gt_results_path, e)
            return None

    size_rng = random.Random(RANDOM_SEED + N)
    size_records = list(records_pool) if N >= len(records_pool) else size_rng.sample(records_pool, N)
    logger.info("[%s N=%d] Records: %d", label, N, len(size_records))

    models = build_models_fn()

    # --- Case 2 / Case 3 (ckpt present) or Case 4 (missing -> fresh) ---
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        start_epoch = ckpt["epoch"]
        for key in state_dict_keys:
            models[key].load_state_dict(ckpt[key])
        models["optimizer"].load_state_dict(ckpt["optimizer"])
        global_step = ckpt["global_step"]
        history = ckpt.get("history", {"epoch": []})
        if start_epoch >= target_epochs:
            logger.info("=" * 70)
            logger.info("=== %s N=%d: checkpoint at target (%d/%d) -- finalizing without training ===",
                        label, N, start_epoch, target_epochs)
            logger.info("=" * 70)
        else:
            logger.info("=" * 70)
            logger.info("=== %s N=%d: resuming epoch %d -> target %d ===", label, N, start_epoch, target_epochs)
            logger.info("=" * 70)
        logger.info("[%s N=%d] Loaded: epoch %d, step %d, history keys=%s",
                    label, N, start_epoch, global_step, list(history.keys()))
    else:
        start_epoch = 0
        global_step = 0
        history = {"epoch": []}
        logger.info("=" * 70)
        logger.info("=== %s N=%d: NO checkpoint at %s -- training FRESH from epoch 0 -> target %d ===",
                    label, N, ckpt_path, target_epochs)
        logger.info("=" * 70)

    def save_checkpoint(epoch, global_step, history):
        tmp_path = ckpt_path + ".tmp"
        state = {key: models[key].state_dict() for key in state_dict_keys}
        state["optimizer"] = models["optimizer"].state_dict()
        state["epoch"] = epoch
        state["global_step"] = global_step
        state["history"] = history
        torch.save(state, tmp_path)
        os.replace(tmp_path, ckpt_path)

    training_start_time = time.time()
    stop_early = False

    try:
        for epoch in range(start_epoch, target_epochs):
            epoch_losses = []
            batches = list(make_batches(size_records, batch_size))
            n_steps = len(batches)

            for step in range(n_steps):
                step_start = time.time()
                models["optimizer"].zero_grad()
                loss = loss_fn(batches[step], models)
                loss.backward()
                models["optimizer"].step()
                global_step += 1
                epoch_losses.append(loss.item())

                elapsed_hours = (time.time() - training_start_time) / 3600
                if step % 500 == 0:
                    logger.info("[%s N=%d] Epoch %d/%d, step %d/%d, loss=%.4f, elapsed=%.2fh",
                                label, N, epoch + 1, target_epochs, step, n_steps, loss.item(), elapsed_hours)

                if global_step % CHECKPOINT_EVERY_N_STEPS == 0:
                    save_checkpoint(epoch, global_step, history)

                if elapsed_hours > MAX_RUNTIME_HOURS_PER_SIZE:
                    logger.warning("[%s N=%d] Runtime limit reached. Saving and stopping.", label, N)
                    save_checkpoint(epoch, global_step, history)
                    stop_early = True
                    break
            if stop_early:
                break

            mean_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
            history["epoch"].append(epoch + 1)
            history.setdefault(loss_key, []).append(mean_loss)
            logger.info("[%s N=%d] Epoch %d done: loss=%.4f", label, N, epoch + 1, mean_loss)
            save_checkpoint(epoch + 1, global_step, history)

            # Per-epoch GT eval + log (always on; matches NB29's per-epoch logging).
            # NB29 does NOT write a CSV per epoch, so neither do we.
            epoch_gt_summary = eval_fn(models)
            logger.info("[%s N=%d] Epoch %d GT results:\n%s",
                        label, N, epoch + 1, epoch_gt_summary.to_string(index=False))

        # --- Final artefacts: match NB29's file layout exactly ---
        pd.DataFrame(history).to_csv(os.path.join(size_dir, "training_history.csv"), index=False)
        for key in state_dict_keys:
            fname_stem = _MODEL_FILE_RENAMES.get(key, key)
            torch.save(models[key].state_dict(), os.path.join(size_dir, f"{fname_stem}.pt"))
        logger.info("[%s N=%d] Saved updated outputs.", label, N)

        gt_summary = eval_fn(models)
        gt_summary.insert(0, "train_size", N)
        gt_summary.to_csv(gt_results_path, index=False)
        logger.info("[%s N=%d] Updated GT results:\n%s", label, N, gt_summary.to_string(index=False))
        return gt_summary

    except torch.cuda.OutOfMemoryError:
        logger.error("[%s N=%d] OOM. Skipping.", label, N)
        return None
    finally:
        try:
            del models
        except NameError:
            pass
        gc.collect()
        if torch.cuda.is_available():
            with torch.cuda.device(CUDA):
                torch.cuda.empty_cache()
                torch.cuda.synchronize()

logger.info("resume_size_to_target_epochs() ready.")

2026-10-24 11:27:02,904 : INFO : resume_size_to_target_epochs() ready.


## Section 8 — Run: image/text large sizes

For each `N` in `IMG_SIZES_TO_RESUME`, the log line at the top of each run tells you whether it took the *skip*, *resume*, *finalize*, or *fresh-start* path. Per-epoch GT results are logged (not saved as CSVs), matching NB29.

In [ ]:
imagetext_results = []
for N in IMG_SIZES_TO_RESUME:
    result = resume_size_to_target_epochs(
        N=N, records_pool=textimage_records, output_dir=IMG_OUTPUT_DIR, batch_size=IMG_BATCH_SIZE,
        build_models_fn=build_models_imagetext, loss_fn=compute_imagetext_loss, eval_fn=evaluate_imagetext,
        state_dict_keys=["head_text_bge", "head_image_dinov3", "dinov3_model", "bge_model"],
        loss_key="imagetext_loss",
        label="ImageText",
    )
    if result is not None:
        imagetext_results.append(result)

logger.info("Image/text resume complete.")

2026-10-23 09:39:43,464 : INFO : [ImageText N=100] Already completed (/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/TextImage_Smaller_train_sizes_runs/size_100/gt_results.csv exists) -- skipping.
2026-10-23 09:39:43,503 : INFO : [ImageText N=500] Already completed (/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/TextImage_Smaller_train_sizes_runs/size_500/gt_results.csv exists) -- skipping.
2026-10-23 09:39:43,511 : INFO : [ImageText N=1000] Already completed (/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/TextImage_Smaller_train_sizes_runs/size_1000/gt_results.csv exists) -- skipping.
2026-10-23 09:39:43,518 : INFO : [ImageText N=2500] Already completed (/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/TextImage_Smaller_train_sizes_runs/size_2500/gt_results.csv exists) -- skipping.
2026-10-23 09:39:43,526 : INFO : [ImageText N=5000] Already completed (/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/TextImage_Smaller_tr

## Section 9 — Run: text/table large sizes

In [10]:
texttable_results = []
for N in TAB_SIZES_TO_RESUME:
    result = resume_size_to_target_epochs(
        N=N, records_pool=texttable_records, output_dir=TAB_OUTPUT_DIR, batch_size=TAB_BATCH_SIZE,
        build_models_fn=build_models_texttable, loss_fn=compute_texttable_loss, eval_fn=evaluate_texttable,
        state_dict_keys=["head_text_bge_v2", "head_table_bge_v2", "bge_model"],
        loss_key="texttable_loss",
        label="TextTable",
    )
    if result is not None:
        texttable_results.append(result)

logger.info("Text/table resume complete.")

2026-10-24 11:27:02,959 : INFO : [TextTable N=100] Already completed (/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/Texttable_Smaller_train_sizes_runs/size_100/gt_results.csv exists) -- skipping.
2026-10-24 11:27:02,981 : INFO : [TextTable N=500] Already completed (/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/Texttable_Smaller_train_sizes_runs/size_500/gt_results.csv exists) -- skipping.
2026-10-24 11:27:02,985 : INFO : [TextTable N=1000] Already completed (/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/Texttable_Smaller_train_sizes_runs/size_1000/gt_results.csv exists) -- skipping.
2026-10-24 11:27:02,988 : INFO : [TextTable N=2500] Already completed (/projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/Texttable_Smaller_train_sizes_runs/size_2500/gt_results.csv exists) -- skipping.
2026-10-24 11:27:02,993 : INFO : [TextTable N=5000] Records: 5000
2026-10-24 11:27:14,110 : INFO : =====================================================

## Section 10 — Combined summary per output dir (notebook 29 convention)

Scans each output dir for `size_{N}/gt_results.csv` and concatenates into `summary_all_sizes.csv` — same filename/purpose as notebook 29's Section 10. All paths are under `OUTPUT_ROOT`.

In [13]:
def collect_summary(output_dirs, label):
    """Scan every `output_dir/size_{N}/gt_results.csv`, concat, dedup by (train_size, direction),
    save to `<first_output_dir>/summary_all_sizes.csv`."""
    rows = []
    for output_dir in output_dirs:
        if not os.path.isdir(output_dir):
            logger.warning("[%s] Output dir not found: %s", label, output_dir)
            continue
        for entry in sorted(os.listdir(output_dir)):
            m = re.match(r"size_(\d+)$", entry)
            if not m:
                continue
            path = os.path.join(output_dir, entry, "gt_results.csv")
            if os.path.exists(path):
                rows.append(pd.read_csv(path))
            else:
                logger.warning("[%s] No gt_results.csv at %s", label, path)
    if not rows:
        logger.warning("[%s] No gt_results.csv files found in %s", label, output_dirs)
        return None
    df = (pd.concat(rows, ignore_index=True)
            .drop_duplicates(subset=["train_size", "direction"], keep="last")
            .sort_values(["train_size", "direction"])
            .reset_index(drop=True))
    out_path = os.path.join(output_dirs[0], "summary_all_sizes.csv")
    df.to_csv(out_path, index=False)
    logger.info("[%s] Combined summary written to %s:\n%s", label, out_path, df.to_string(index=False))
    return df

summary_image_text = collect_summary([IMG_OUTPUT_DIR], "ImageText")
summary_text_table = collect_summary([TAB_OUTPUT_DIR, E3_FULL_CKPT_PARENT_DIR], "TextTable")

try:
    display(summary_image_text)
except NameError:
    print(summary_image_text)
try:
    display(summary_text_table)
except NameError:
    print(summary_text_table)

2026-10-24 20:18:29,633 : INFO : [ImageText] Combined summary written to /projects/agrifutur/sji/cross_encoder_ft/Smaller_train_sizes/TextImage_Smaller_train_sizes_runs/summary_all_sizes.csv:
 train_size     direction   N   R@1   R@3   R@5  R@10      MRR
        100 image -> text 500 0.036 0.082 0.106 0.144 0.078078
        100 text -> image 500 0.032 0.074 0.102 0.132 0.072574
        500 image -> text 500 0.072 0.154 0.196 0.280 0.143341
        500 text -> image 500 0.080 0.152 0.192 0.280 0.145776
       1000 image -> text 500 0.108 0.232 0.280 0.362 0.198925
       1000 text -> image 500 0.110 0.212 0.270 0.368 0.195489
       2500 image -> text 500 0.202 0.370 0.430 0.538 0.313194
       2500 text -> image 500 0.190 0.362 0.436 0.528 0.309185
       5000 image -> text 500 0.296 0.478 0.542 0.650 0.416807
       5000 text -> image 500 0.328 0.494 0.568 0.660 0.438777
      10000 image -> text 500 0.368 0.552 0.624 0.722 0.487637
      10000 text -> image 500 0.378 0.562 0.638 0.73

,train_size,direction,N,R@1,R@3,R@5,R@10,MRR
0,100,image -> text,500,0.036,0.082,0.106,0.144,0.078078
1,100,text -> image,500,0.032,0.074,0.102,0.132,0.072574
2,500,image -> text,500,0.072,0.154,0.196,0.280,0.143341
3,500,text -> image,500,0.080,0.152,0.192,0.280,0.145776
4,1000,image -> text,500,0.108,0.232,0.280,0.362,0.198925
5,1000,text -> image,500,0.110,0.212,0.270,0.368,0.195489
6,2500,image -> text,500,0.202,0.370,0.430,0.538,0.313194
7,2500,text -> image,500,0.190,0.362,0.436,0.528,0.309185
8,5000,image -> text,500,0.296,0.478,0.542,0.650,0.416807
9,5000,text -> image,500,0.328,0.494,0.568,0.660,0.438777


,train_size,direction,N,R@1,R@3,R@5,R@10,MRR
0,100,table -> text,500,0.070,0.140,0.168,0.260,0.135943
1,100,text -> table,500,0.070,0.124,0.170,0.256,0.130646
2,500,table -> text,500,0.150,0.314,0.422,0.532,0.271755
3,500,text -> table,500,0.130,0.324,0.422,0.518,0.262667
4,1000,table -> text,500,0.206,0.400,0.524,0.630,0.345791
5,1000,text -> table,500,0.218,0.442,0.534,0.632,0.362474
6,2500,table -> text,500,0.296,0.528,0.634,0.774,0.445214
7,2500,text -> table,500,0.316,0.542,0.648,0.748,0.462810
8,5000,table -> text,500,0.388,0.636,0.716,0.828,0.537372
9,5000,text -> table,500,0.364,0.638,0.752,0.838,0.529283
